In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings('ignore')

PARQUET_PATH = 'signals.parquet'
OUTPUT_DIR   = 'prepared_data/v2_16384'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLING_RATE   = 51200   # signalo daznis
SEGMENT_SAMPLES = 16384   # norimo segmento ilgis

COL_FEATURE  = 'Feature'
COL_BANDYMAS = 'Bandymas'
COL_APKROVA  = 'Apkrova(Nm)'
COL_SUKIAI   = 'Sukiai(rpm)'
COL_SIGNAL   = 'Value'

VAL_SIZE     = 0.2
RANDOM_STATE = 42

print(f'Segmento ilgis : {SEGMENT_SAMPLES} reikšmių'
      f'({1000 * SEGMENT_SAMPLES / SAMPLING_RATE:.1f} ms)')
print(f'Kiek gaunasi segmentu per visa signala: {256000 // SEGMENT_SAMPLES}')

## 1. Parquet failo uzkrovimas ir signalu rekonstravimas

In [ ]:
print('Užkraunamas parquet failas...')
df = pd.read_parquet(PARQUET_PATH)
df[COL_FEATURE]  = df[COL_FEATURE].str.replace('Feat', '').astype(int)
df[COL_BANDYMAS] = df[COL_BANDYMAS].str.extract(r'(\d+)').astype(int)
print(f'Loaded: {df.shape[0]:,} rows')

print('Rekonstruojami signalai...')
signal_groups = df.groupby(
    [COL_FEATURE, COL_BANDYMAS, COL_APKROVA, COL_SUKIAI]
)[COL_SIGNAL].apply(np.array).reset_index()
signal_groups.columns = ['label', 'batch', 'load', 'rpm', 'signal']

print(f'Signalu skaicius : {len(signal_groups)}')
print(f'Signalo ilgis : {len(signal_groups.iloc[0]["signal"]):,} samples')
print(f'\nSignalu per gedima:')
print(signal_groups.groupby('label').size().to_string())
print(f'\nSignalu per bandyma:')
print(signal_groups.groupby('batch').size().to_string())

## 2. Signalų skaidymas

In [ ]:
def segment_signals(signal_df, segment_samples):
    segments, labels, batches = [], [], []

    for _, row in signal_df.iterrows():
        sig   = row['signal'].astype(np.float32)
        label = int(row['label'])
        batch = int(row['batch'])

        n_segs = len(sig) // segment_samples
        for i in range(n_segs):
            window = sig[i * segment_samples : (i + 1) * segment_samples]
            segments.append(window)
            labels.append(label)
            batches.append(batch)

    return (np.stack(segments),
            np.array(labels,  dtype=np.int64),
            np.array(batches, dtype=np.int64))


print('Segmentuojama...')
X_all, y_all, batch_all = segment_signals(signal_groups, SEGMENT_SAMPLES)

print(f'\nIs viso segmentu : {len(X_all):,}')
print(f'Segmentu skaicius ir segmento ilgis  : {X_all.shape}')
print(f'\nPasiskirstymas pagal gedima (visi segmentai):')
unique, counts = np.unique(y_all, return_counts=True)
for c, n in zip(unique, counts):
    print(f'  Gedimas {c}: {n:,}  ({100*n/len(y_all):.1f}%)')

## 3. Treniravimo/validavimo/testavimo aibiu sudarymas

In [ ]:
test_mask  = batch_all == 2
train_mask = batch_all == 1

X_test, y_test = X_all[test_mask], y_all[test_mask]
X_b1,   y_b1   = X_all[train_mask], y_all[train_mask]

X_train, X_val, y_train, y_val = train_test_split(
    X_b1, y_b1,
    test_size    = VAL_SIZE,
    stratify     = y_b1,
    random_state = RANDOM_STATE
)

print(f'X_train : {X_train.shape}  y_train : {y_train.shape}')
print(f'X_val   : {X_val.shape}  y_val   : {y_val.shape}')
print(f'X_test  : {X_test.shape}  y_test  : {y_test.shape}')

print(f'\nTreniravimo segmentu pasiskirstymas:')
unique, counts = np.unique(y_train, return_counts=True)
for c, n in zip(unique, counts):
    print(f'  Class {c}: {n:,}  ({100*n/len(y_train):.1f}%)')

## 4. Segmentu vizualizavimas

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for cls in range(7):
    idx = np.where(y_train == cls)[0][0]
    axes[cls].plot(X_train[idx], linewidth=0.8)
    axes[cls].set_title(f'Class {cls}', fontsize=10)
    axes[cls].set_xlabel('Sample')
    axes[cls].set_ylabel('Amplitude')
    axes[cls].grid(True, alpha=0.3)

axes[7].axis('off')
plt.suptitle(f'Segmentu pavyzdziai (ilgis={SEGMENT_SAMPLES})', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Issaugojimas i failus

In [ ]:
print('Issaugojama...')

np.save(f'{OUTPUT_DIR}/X_train.npy', X_train)
np.save(f'{OUTPUT_DIR}/X_val.npy',   X_val)
np.save(f'{OUTPUT_DIR}/X_test.npy',  X_test)
np.save(f'{OUTPUT_DIR}/y_train.npy', y_train)
np.save(f'{OUTPUT_DIR}/y_val.npy',   y_val)
np.save(f'{OUTPUT_DIR}/y_test.npy',  y_test)

print(f'\nFailai issaugoti: {OUTPUT_DIR}/')
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.npy'):
        size_mb = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1e6
        print(f'  {f:<25} {size_mb:.1f} MB')